# ⛓️ Module 04: LCEL — LangChain Expression Language

---

## What is LCEL?

**LCEL (LangChain Expression Language)** is the core composability layer of LangChain. It uses the **`|` pipe operator** to chain components together — inspired by Unix pipes.

```python
# Every chain is just piped components!
chain = prompt | llm | parser
result = chain.invoke(input)
```

### Why LCEL?

| Feature | Benefit |
|---------|----------|
| **Streaming** | Built-in streaming for every chain |
| **Async** | Native async support |
| **Parallel** | Run branches simultaneously |
| **Batch** | Process multiple inputs at once |
| **Tracing** | Automatic LangSmith integration |
| **Fallbacks** | Add fallback chains on error |
| **Type safety** | Input/output schemas at every step |

---

## The Runnable Interface

Everything in LCEL implements `Runnable`:

```
Runnable
├── .invoke(input)     → single output
├── .stream(input)     → generator of chunks
├── .batch(inputs)     → list of outputs
├── .ainvoke(input)    → async single output
├── .astream(input)    → async generator
└── .abatch(inputs)    → async list of outputs
```

---

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)
print("Setup complete ✅")

Setup complete ✅


## 1️⃣ Basic Chain Composition

In [2]:
# ============================================================
# The simplest chain: prompt | llm | parser
# ============================================================
prompt = ChatPromptTemplate.from_template(
    "Explain {concept} in exactly 2 sentences."
)
parser = StrOutputParser()

# Compose with pipe operator
chain = prompt | llm | parser

# The chain is a Runnable — call it with invoke
result = chain.invoke({"concept": "neural networks"})
print("Type:", type(result).__name__)
print("Result:", result)

Type: TextAccessor
Result: Neural networks are a type of machine learning model inspired by the structure and function of the human brain, consisting of interconnected nodes (neurons) that process and transmit information through complex patterns of activation and inhibition. By training on large datasets, neural networks can learn to recognize patterns, make predictions, and classify inputs, allowing them to perform tasks such as image recognition, natural language processing, and decision-making.


In [3]:
# ============================================================
# Inspect chain input/output schema
# ============================================================
print("Input schema:")
print(chain.input_schema.model_json_schema())

print("\nOutput schema:")
print(chain.output_schema.model_json_schema())

Input schema:
{'properties': {'concept': {'title': 'Concept', 'type': 'string'}}, 'required': ['concept'], 'title': 'PromptInput', 'type': 'object'}

Output schema:
{'title': 'StrOutputParserOutput', 'type': 'string'}


In [4]:
# ============================================================
# Streaming through the chain
# ============================================================
print("Streaming output:")
print("-" * 50)

for chunk in chain.stream({"concept": "quantum computing"}):
    print(chunk, end="", flush=True)

print("\n" + "-" * 50)

Streaming output:
--------------------------------------------------
Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform calculations and operations on data, enabling it to process vast amounts of information exponentially faster than classical computers by harnessing the power of superposition, entanglement, and interference. This allows quantum computers to solve complex problems in fields like cryptography, optimization, and simulation that are beyond the capabilities of traditional computers, with potential applications in fields such as medicine, finance, and climate modeling.
--------------------------------------------------


In [5]:
# ============================================================
# Batch processing
# ============================================================
concepts = [
    {"concept": "recursion"},
    {"concept": "REST API"},
    {"concept": "Docker containers"},
]

results = chain.batch(concepts)

for concept, result in zip(concepts, results):
    print(f"\n📘 {concept['concept'].upper()}:")
    print(f"   {result}")


📘 RECURSION:
   Recursion is a programming technique where a function calls itself repeatedly until it reaches a base case that stops the recursion, allowing the function to return values and unwind the call stack. This process allows the function to solve complex problems by breaking them down into smaller, more manageable sub-problems that are identical in structure, making it a powerful tool for solving problems with recursive structures.

📘 REST API:
   A REST API (Representational State of Resource) is an architectural style for designing networked applications, where resources are identified by URIs, manipulated using a fixed set of operations (HTTP methods), and communicated in a stateless and cacheable manner. This allows for a flexible and scalable way to interact with web services, enabling different clients to access and manipulate data in a standardized and platform-independent way.

📘 DOCKER CONTAINERS:
   Docker containers are lightweight and portable packages that encap

## 2️⃣ RunnableLambda — Wrap Any Function

In [6]:
from langchain_core.runnables import RunnableLambda

# ============================================================
# Turn any Python function into a Runnable
# ============================================================

def word_counter(text: str) -> dict:
    """Count words and characters in text"""
    words = text.split()
    return {
        "text": text,
        "word_count": len(words),
        "char_count": len(text),
        "avg_word_length": sum(len(w) for w in words) / len(words) if words else 0
    }

def format_stats(stats: dict) -> str:
    """Format stats into a readable string"""
    return f"""
Text: "{stats['text'][:50]}..."
Words: {stats['word_count']}
Characters: {stats['char_count']}
Avg word length: {stats['avg_word_length']:.1f}
"""

# Chain: prompt -> llm -> StrOutputParser -> word_counter -> format_stats
analysis_chain = (
    ChatPromptTemplate.from_template("Write a paragraph about {topic}")
    | llm
    | StrOutputParser()
    | RunnableLambda(word_counter)
    | RunnableLambda(format_stats)
)

result = analysis_chain.invoke({"topic": "black holes"})
print(result)


Text: "Black holes are among the most mysterious and fasc..."
Words: 150
Characters: 918
Avg word length: 5.1



In [7]:
# ============================================================
# Lambda shorthand — Use Python lambdas directly!
# ============================================================
chain = (
    ChatPromptTemplate.from_template("List 5 {category} in a comma-separated list, nothing else.")
    | llm
    | StrOutputParser()
    | (lambda text: [item.strip() for item in text.split(',')])  # Direct lambda!
    | (lambda items: {"items": items, "count": len(items)})
)

result = chain.invoke({"category": "programming languages"})
print("Result:", result)

Result: {'items': ['Java', 'Python', 'JavaScript', 'C++', 'Ruby'], 'count': 5}


## 3️⃣ RunnableParallel — Run Branches Simultaneously

In [8]:
from langchain_core.runnables import RunnableParallel

# ============================================================
# Run multiple chains at the same time
# ============================================================

# Three different analysis chains
summary_chain = (
    ChatPromptTemplate.from_template("Summarize this topic in 1 sentence: {topic}")
    | llm | StrOutputParser()
)

pros_chain = (
    ChatPromptTemplate.from_template("List 3 pros of {topic} in bullet points")
    | llm | StrOutputParser()
)

cons_chain = (
    ChatPromptTemplate.from_template("List 3 cons of {topic} in bullet points")
    | llm | StrOutputParser()
)

# RunnableParallel runs all three SIMULTANEOUSLY
parallel_analysis = RunnableParallel(
    summary=summary_chain,
    pros=pros_chain,
    cons=cons_chain
)

import time
start = time.time()
result = parallel_analysis.invoke({"topic": "TypeScript vs JavaScript"})
elapsed = time.time() - start

print(f"Completed in {elapsed:.2f}s (would take ~{elapsed*3:.0f}s sequentially)")
print()
print("📋 SUMMARY:")
print(result['summary'])
print("\n✅ PROS:")
print(result['pros'])
print("\n❌ CONS:")
print(result['cons'])

Completed in 0.77s (would take ~2s sequentially)

📋 SUMMARY:
TypeScript and JavaScript are two closely related programming languages, with TypeScript being a superset of JavaScript that adds optional static typing and other features to improve code maintainability, scalability, and reliability.

✅ PROS:
Here are three pros of TypeScript compared to JavaScript:

* **Stronger Type System**: TypeScript has a statically-typed type system, which helps catch errors early in the development process. This reduces the likelihood of runtime errors and makes the code more maintainable. In contrast, JavaScript is dynamically-typed, which can lead to type-related issues at runtime.
* **Better Code Completion and Refactoring**: The type information in TypeScript allows for better code completion and refactoring in integrated development environments (IDEs) and text editors. This makes it easier for developers to write and maintain large codebases.
* **Improved Scalability and Maintainability**: Type

In [9]:
# ============================================================
# Parallel with passthrough — Keep original input
# ============================================================
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chain_with_passthrough = RunnableParallel(
    original=RunnablePassthrough(),   # Pass input through unchanged
    improved=(
        ChatPromptTemplate.from_template("Improve this text: {text}")
        | llm | StrOutputParser()
    )
)

result = chain_with_passthrough.invoke({"text": "the cat sat on mat"})
print("Original:", result['original'])
print("Improved:", result['improved'])

Original: {'text': 'the cat sat on mat'}
Improved: Here's an improved version of the text:

The cat sat comfortably on the soft mat.

I made a few changes to enhance the text:

- Added a descriptive word "comfortably" to give a clearer image of the cat's position.
- Added a descriptive word "soft" to describe the mat, which adds more detail to the scene.
- Changed the phrase "on mat" to "on the soft mat" to make the sentence more grammatically correct and easier to read.


## 4️⃣ RunnablePassthrough — Pass Input Unchanged

In [10]:
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# Classic RAG pattern using RunnablePassthrough
# ============================================================
# Pattern: We need both the original question AND the retrieved context

def fake_retriever(query: str) -> str:
    """Simulates retrieving relevant documents"""
    return "LangChain was created by Harrison Chase in 2022. It is open source."

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])

rag_chain = (
    RunnableParallel(
        context=RunnableLambda(fake_retriever),  # Get context
        question=RunnablePassthrough()           # Pass question through
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Who created LangChain?")
print("Answer:", answer)

Answer: Harrison Chase created LangChain in 2022.


## 5️⃣ RunnableBranch — Conditional Logic

In [11]:
from langchain_core.runnables import RunnableBranch

# ============================================================
# Route to different chains based on input
# ============================================================

# Different response chains
technical_chain = (
    ChatPromptTemplate.from_template("Provide a technical, detailed answer to: {question}")
    | llm | StrOutputParser()
)

simple_chain = (
    ChatPromptTemplate.from_template("Explain simply for a beginner: {question}")
    | llm | StrOutputParser()
)

default_chain = (
    ChatPromptTemplate.from_template("Answer this general question: {question}")
    | llm | StrOutputParser()
)

# Branch based on question content
branch = RunnableBranch(
    # (condition, chain_to_use)
    (lambda x: "code" in x["question"].lower() or "algorithm" in x["question"].lower(), technical_chain),
    (lambda x: "simple" in x["question"].lower() or "beginner" in x["question"].lower(), simple_chain),
    default_chain  # Default fallback
)

# Test with different questions
questions = [
    {"question": "How does the quicksort algorithm work?"},
    {"question": "Simple explanation of what Python is for a beginner?"},
    {"question": "What is the history of programming?"}
]

for q in questions:
    print(f"Question: {q['question']}")
    print(f"Answer: {branch.invoke(q)[:100]}...\n")

Question: How does the quicksort algorithm work?
Answer: **Quicksort Algorithm Overview**

Quicksort is a popular, efficient, and widely used sorting algorit...

Question: Simple explanation of what Python is for a beginner?
Answer: **What is Python?**

Python is a programming language that helps you tell a computer what to do. It'...

Question: What is the history of programming?
Answer: The history of programming is a long and complex one, spanning over a century. It involves the devel...



## 6️⃣ Chain Fallbacks — Handle Failures Gracefully

In [ ]:
from langchain_groq import ChatGroq
from langchain_anthropic import ChatAnthropic

# ============================================================
# Primary chain uses llama -70B fallback uses llama-8b
# ============================================================
primary_llm = ChatGroq(model="llama-3.3-70b-versatile")         # More expensive/powerful
fallback_llm = ChatGroq(model="llama-3.1-8b-instant")   # Cheaper fallback

prompt = ChatPromptTemplate.from_template("Answer: {question}")
parser = StrOutputParser()

primary_chain = prompt | primary_llm | parser
fallback_chain = prompt | fallback_llm | parser

# Chain with fallback
robust_chain = primary_chain.with_fallbacks([fallback_chain])

# This will try primary first, then fallback on error
result = robust_chain.invoke({"question": "What is 42?"})
print("Result:", result)

Result: The answer "42" is a reference to Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, a supercomputer named Deep Thought is asked to find the "Answer to the Ultimate Question of Life, the Universe, and Everything." After thinking for 7.5 million years, Deep Thought reveals that the answer is indeed 42.

However, the book never explicitly states what the ultimate question is, leaving the reader to ponder the meaning of the answer. The number 42 has since become a popular cultural reference, often used humorously or ironically to represent a mysterious or unknowable answer to a complex question.

In reality, the number 42 doesn't hold any specific significance or mathematical property that would make it a meaningful answer to a profound question. It's simply a clever literary device created by Douglas Adams to satirize the idea of seeking simple answers to complex problems.


## 7️⃣ Chaining Chains — Sequential Multi-Step Workflows

In [13]:
# ============================================================
# Multi-step content pipeline
# Step 1: Generate an outline
# Step 2: Write a blog post from that outline
# Step 3: Create a catchy title for the post
# ============================================================

# Step 1: Outline chain
outline_chain = (
    ChatPromptTemplate.from_template("Create a 5-point outline for a blog post about {topic}")
    | llm | StrOutputParser()
)

# Step 2: Blog post chain (takes outline as input)
blog_chain = (
    ChatPromptTemplate.from_template(
        "Write a 200-word blog post based on this outline:\n{outline}"
    )
    | llm | StrOutputParser()
)

# Step 3: Title chain (takes blog post as input)
title_chain = (
    ChatPromptTemplate.from_template(
        "Generate 3 catchy SEO-friendly titles for this blog post:\n{blog_post}"
    )
    | llm | StrOutputParser()
)

# Connect them: the output of each becomes the input of the next
full_pipeline = (
    {"topic": RunnablePassthrough()}   # Input
    | RunnableParallel(
        outline=outline_chain,
        topic=RunnablePassthrough()    # Keep topic for reference
    )
    | {
        "blog_post": ({"outline": lambda x: x["outline"]} | blog_chain),
        "outline": lambda x: x["outline"]
    }
    | RunnableParallel(
        blog_post=lambda x: x["blog_post"],
        titles=({"blog_post": lambda x: x["blog_post"]} | title_chain)
    )
)

result = full_pipeline.invoke({"topic": "the future of AI in healthcare"})

print("✍️ BLOG POST:")
print(result["blog_post"])
print("\n📌 TITLE OPTIONS:")
print(result["titles"])

✍️ BLOG POST:
**The Future of AI in Healthcare: Revolutionizing Diagnosis, Treatment, and Patient Care**

The healthcare industry is on the cusp of a revolution, driven by the rapid advancement of artificial intelligence (AI) technologies. AI is transforming the way healthcare professionals diagnose, treat, and care for patients, with potential applications in every aspect of healthcare. From AI-powered diagnostic tools to personalized medicine and patient engagement platforms, the future of AI in healthcare holds immense promise.

**AI-Powered Diagnosis and Detection**

AI algorithms and machine learning techniques are being used to analyze medical images, such as X-rays, CT scans, and MRIs, to detect diseases at an early stage. Computer-aided detection (CAD) systems, for example, can help doctors identify cancerous tumors with greater accuracy. AI-powered diagnostic tools can also reduce false positives and negatives, leading to more effective treatment plans.

**Personalized Medicin

## 8️⃣ Inspecting Chains with astream_events

In [14]:
# ============================================================
# astream_events — See what's happening at each step
# ============================================================

chain = (
    ChatPromptTemplate.from_template("Write a one-sentence fact about {topic}")
    | llm
    | StrOutputParser()
)

print("Events from chain execution:")
print("-" * 50)

async for event in chain.astream_events({"topic": "Mars"}, version="v2"):
    kind = event["event"]
    name = event.get("name", "")
    
    if kind == "on_chain_start":
        print(f"🟢 START: {name}")
    elif kind == "on_chain_end":
        print(f"🔴 END: {name}")
    elif kind == "on_chat_model_stream":
        chunk = event["data"]["chunk"].content
        if chunk:
            print(chunk, end="", flush=True)

print("\n" + "-" * 50)

Events from chain execution:
--------------------------------------------------
🟢 START: RunnableSequence
Mars, often referred to as the Red Planet, has the tallest volcano in our solar system, Olympus Mons, which stands at an impressive height of approximately 27 km (17 miles) above the Martian surface.🔴 END: RunnableSequence

--------------------------------------------------


## ✅ Module 04 Summary

You've learned:
- ✅ LCEL pipe `|` operator for chain composition
- ✅ The Runnable interface (invoke, stream, batch, async)
- ✅ `RunnableLambda` — wrap any Python function
- ✅ `RunnableParallel` — run branches simultaneously
- ✅ `RunnablePassthrough` — pass input unchanged
- ✅ `RunnableBranch` — conditional routing
- ✅ Fallbacks for error resilience
- ✅ Multi-step sequential workflows

### 🚀 Next: [Module 05 — Memory & Conversation History](05_Memory_and_Conversations.ipynb)